## 00_setup_sample_data  (OLD workspace = Databricks Free Edition)

Creates the practice tables used by the jobs and the join playground (17):
**10 relational base tables** (customer-360 star schema: customers hub +
addresses, agents, products, campaigns, accounts, subscriptions, transactions,
support_tickets, campaign_responses) + 4 job-derived tables (clean, enriched,
summary, predictions) = 14 lifted by 35 -> 47.
Zero cost: ~1k rows total, pure SQL, no external data, no downloads.
OLD workspace = Databricks Free Edition (serverless, Unity Catalog):
  - table names resolved from the SCHEMA widget only (old workspace: schema.table, no catalog
    — SCHEMA is the OLD schema name, kept IDENTICAL after migration)
  - classic fallback workspaces (Community Edition / DBR 12.2) also work —
    the code auto-detects 2-part vs 3-part naming
  - every table is partitioned by ingest_date (dates spread 2026-10-01..10)
  - a PARTITION widget is read by every job task, not by this setup notebook
Run manually once per environment (old workspace, then new workspace).


In [ ]:


import re
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (the OLD schema name — kept identical after migration)")

SCHEMA = dbutils.widgets.get("SCHEMA").strip()


def _identifier(value):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value or ""), f"Invalid identifier: {value!r}"
    return value


def table_name(simple_name):
    return f"{_identifier(SCHEMA)}.{_identifier(simple_name)}"


In [ ]:


target_schema = _identifier(SCHEMA)
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")
except Exception as e:
    print(f"Schema auto-create skipped ({str(e)[:120]}...) — assuming it already exists")


In [ ]:


# 100 synthetic customers, fully deterministic (seed 42) so old vs new
# results can be compared exactly. ingest_date spreads rows over 10 dates.
spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_customers')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                          AS customer_id,
    CASE
        WHEN id % 3 = 0 THEN 'retail'
        WHEN id % 3 = 1 THEN 'enterprise'
        ELSE 'smb'
    END                                      AS segment,
    CAST(20 + 980 * rand(42 + id) AS DOUBLE) AS monthly_revenue,
    CAST(rand(99 + id) AS DOUBLE)            AS churn_risk,
    CAST(id % 12 AS INT)                     AS tenure_months,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(100)
""")

tbl = table_name("practice_customers")
print(f"Table created: {tbl}")
display(spark.table(tbl).groupBy("ingest_date").count().orderBy("ingest_date"))


In [ ]:

# Base tables 2/10 to 10/10 — customer-360 relational star schema
# (addresses, agents, products, campaigns, accounts, subscriptions,
#  transactions, support_tickets, campaign_responses). Same rules as
# practice_customers: deterministic (seeded), partitioned by ingest_date,
# 2026-10-01..10. Every FK is id % <parent_count>, so all join keys resolve;
# a few children (tickets 80, responses 60) intentionally do NOT cover every
# customer — that is what makes LEFT vs INNER joins show real differences.

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_addresses')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                                     AS address_id,
    CAST(id % 100 AS INT)                               AS customer_id,
    CASE WHEN id % 2 = 0 THEN 'home' ELSE 'billing' END AS address_type,
    CONCAT('city_', CAST(id % 15 AS INT))               AS city,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(120)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_agents')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                       AS agent_id,
    CONCAT('agent_', CAST(id AS STRING))  AS agent_name,
    CONCAT('region_', CAST(id % 4 AS INT)) AS region,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(10)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_products')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                          AS product_id,
    CONCAT('product_', CAST(id AS STRING))   AS product_name,
    CONCAT('cat_', CAST(id % 5 AS INT))      AS category,
    CAST(5 + 45 * rand(7 + id) AS DOUBLE)    AS unit_price,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(20)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_campaigns')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                          AS campaign_id,
    CONCAT('campaign_', CAST(id AS STRING))  AS campaign_name,
    CASE id % 3 WHEN 0 THEN 'email' WHEN 1 THEN 'sms' ELSE 'push' END AS channel,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(5)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_accounts')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                              AS account_id,
    CAST(id % 100 AS INT)                        AS customer_id,
    CASE WHEN id % 2 = 0 THEN 'checking' ELSE 'savings' END AS account_type,
    CAST(100 + 4900 * rand(11 + id) AS DOUBLE)   AS balance,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(130)
""")

print("Base tables 2-6 created: addresses, agents, products, campaigns, accounts")


In [ ]:

# Base tables 7/10 to 10/10 — bridges + fact table.

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_subscriptions')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                               AS subscription_id,
    CAST(id % 130 AS INT)                         AS account_id,
    CAST(id % 20 AS INT)                          AS product_id,
    CAST(CONCAT('2026-0', CAST(1 + (id % 9) AS STRING), '-01') AS DATE) AS started_on,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(150)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_transactions')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                               AS txn_id,
    CAST(id % 130 AS INT)                         AS account_id,
    CAST(id % 20 AS INT)                          AS product_id,
    CAST(1 + 99 * rand(23 + id) AS DOUBLE)        AS amount,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS txn_date,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(400)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_support_tickets')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                               AS ticket_id,
    CAST(id % 100 AS INT)                         AS customer_id,
    CAST(id % 10 AS INT)                          AS agent_id,
    CASE id % 3 WHEN 0 THEN 'open' WHEN 1 THEN 'pending' ELSE 'closed' END AS status,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(80)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {table_name('practice_campaign_responses')}
USING DELTA
PARTITIONED BY (ingest_date)
AS
SELECT
    CAST(id AS INT)                               AS response_id,
    CAST(id % 100 AS INT)                         AS customer_id,
    CAST(id % 5 AS INT)                           AS campaign_id,
    CAST(id % 2 = 0 AS BOOLEAN)                   AS responded,
    CAST(CONCAT('2026-10-', LPAD(CAST(1 + (id % 10) AS STRING), 2, '0')) AS DATE) AS ingest_date
FROM range(60)
""")

print("Base tables 7-10 created: subscriptions, transactions, support_tickets, campaign_responses")


In [ ]:

# Sanity — row counts + referential integrity (every DISTINCT child key must
# resolve in its parent; orphaned keys would silently shrink INNER joins).

fk_checks = [
    ("practice_addresses", "customer_id", "practice_customers", "customer_id"),
    ("practice_accounts", "customer_id", "practice_customers", "customer_id"),
    ("practice_subscriptions", "account_id", "practice_accounts", "account_id"),
    ("practice_subscriptions", "product_id", "practice_products", "product_id"),
    ("practice_transactions", "account_id", "practice_accounts", "account_id"),
    ("practice_transactions", "product_id", "practice_products", "product_id"),
    ("practice_support_tickets", "customer_id", "practice_customers", "customer_id"),
    ("practice_support_tickets", "agent_id", "practice_agents", "agent_id"),
    ("practice_campaign_responses", "customer_id", "practice_customers", "customer_id"),
    ("practice_campaign_responses", "campaign_id", "practice_campaigns", "campaign_id"),
]

rows = []
for child, fk, parent, pk in fk_checks:
    orphan = spark.sql(f"""
        SELECT COUNT(*) AS n FROM (
            SELECT DISTINCT {fk} FROM {table_name(child)}
        ) c
        LEFT ANTI JOIN {table_name(parent)} p ON c.{fk} = p.{pk}
    """).collect()[0]["n"]
    assert orphan == 0, f"{child}.{fk}: {orphan} orphaned key(s) not in {parent}.{pk}"
    rows.append({"table": table_name(child), "rows": spark.table(table_name(child)).count(),
                 "fk -> parent": f"{fk} -> {parent}"})

base_tables = ["practice_customers", "practice_addresses", "practice_accounts",
               "practice_products", "practice_subscriptions", "practice_transactions",
               "practice_support_tickets", "practice_agents", "practice_campaigns",
               "practice_campaign_responses"]
for t in base_tables:
    if not any(r["table"].endswith(t) for r in rows):
        rows.append({"table": table_name(t), "rows": spark.table(table_name(t)).count(), "fk -> parent": "(hub/dimension)"})

import pandas as pd
summary = pd.DataFrame(rows).sort_values("table").reset_index(drop=True)
display(summary)
print(f"10/10 base tables ready in {target_schema} — all FKs resolve (0 orphans).")
print("Derived tables (clean/enriched/summary/predictions) come from the practice jobs (Phase 1.3).")


In [ ]:


# The table chosen for model training (notebooks 01-05 read this).
# features: tenure_months, churn_risk, seg_enterprise, seg_retail
# label:    monthly_revenue
TRAIN_TABLE = table_name("practice_customers")
display(spark.table(TRAIN_TABLE).orderBy("customer_id").limit(10))
print(f"TRAINING TABLE for notebooks 01-05: {TRAIN_TABLE}")
print("features: tenure_months, churn_risk, seg_enterprise, seg_retail | label: monthly_revenue")
